In [5]:

import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain.agents import create_agent


# 1. Load API key using getpass

GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


# 2. Initialize Gemini LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


# 3. Create Simple QA Tool

qa_prompt = PromptTemplate.from_template(
    "Answer clearly: {question}"
)

def run_qa(question):
    response = (qa_prompt | llm).invoke({
        "question": question
    })

    return response.content


qa_tool = Tool(
    name="Simple_QA",
    func=run_qa,
    description="Answer factual questions clearly."
)


# 4. Create Agent

agent = create_agent(
    model=llm,
    tools=[qa_tool],
    system_prompt="""
    You are a helpful AI assistant.

    Use the Simple_QA tool to answer factual questions clearly.
    Provide accurate and understandable answers.
    """
)


# 5. Run Agent with Conversation History

conversation_history = []

while True:

    query = input("\nYou: ")

    if query.lower().strip() in ["exit", "quit", "bye"]:
        print("\nAssistant: Goodbye!")
        break

    try:

        response = agent.invoke({
            "messages": conversation_history + [
                {"role": "user", "content": query}
            ]
        })

        conversation_history = response["messages"]

        print("\nAssistant:")
        print(response["messages"][-1].content)

    except Exception as e:
        print("\nError:", str(e))


Enter Gemini API Key: ··········

You: what is langh chain and lang graph 

Assistant:
[{'type': 'text', 'text': 'Here is a clear breakdown of **LangChain** and **LangGraph**, what they do, and how they work together.\n\n---\n\n### 1. What is LangChain?\n**LangChain** is an open-source framework designed to help developers build applications powered by Large Language Models (LLMs like ChatGPT, Claude, Llama, etc.).\n\nInstead of just sending a single prompt to an AI and getting a response, LangChain allows you to **"chain"** together different components to build full software applications.\n\n* **Core Concept:** **Linear Workflows** (Step A $\\rightarrow$ Step B $\\rightarrow$ Step C).\n* **What it does:** Connects LLMs to external data sources (PDFs, databases, web pages) and tools (APIs, search engines, calculators).\n* **Best Used For:**\n  * **RAG (Retrieval-Augmented Generation):** "Chat with your PDF" or searching company documentation.\n  * Simple QA chatbots.\n  * Text summari

In [3]:
%pip install -q langchain-google-genai==4.4.3

In [6]:

from langchain.agents import create_agent


# 1. Initialize conversation history

conversation_history = []


# 2. Create agent with QA tool

agent = create_agent(
    model=llm,
    tools=[qa_tool],
    system_prompt="""
    You are a helpful conversational AI assistant.

    Instructions:
    - Answer questions clearly and accurately.
    - Use the Simple_QA tool for factual questions.
    - Remember previous conversation context.
    - Understand follow-up questions using chat history.
    - Explain concepts in simple language.
    """
)


# 3. Define a function to run the conversation

def chat_with_agent(query):

    global conversation_history

    response = agent.invoke({
        "messages": conversation_history + [
            {"role": "user", "content": query}
        ]
    })

    # Update conversation history
    conversation_history = response["messages"]

    return response["messages"][-1].content


# 4. Run multiple interactions

print("1️  First Question")

res1 = chat_with_agent("What is LangChain?")
print("\nAnswer:", res1)


print("\n 2️ Follow-up Question")

res2 = chat_with_agent("Who created it?")
print("\nAnswer:", res2)


print("\n 3️ Ask again about previous topic")

res3 = chat_with_agent("Explain it simply again.")
print("\nAnswer:", res3)


# 5. Display conversation history

print("\nConversation History")

for msg in conversation_history:
    if msg.type in ["human", "ai"]:
        print(f"{msg.type.upper()}: {msg.content}")


1️  First Question

Answer: [{'type': 'text', 'text': '**LangChain** is an open-source framework designed to help developers build applications powered by Large Language Models (LLMs), such as OpenAI’s GPT models, Google Gemini, or open-source models like Llama.\n\n---\n\n### Why is LangChain useful?\nOn their own, LLMs are primarily text generators. They don\'t have access to your private files, can\'t directly search the web or run code, and don\'t remember past conversations natively. \n\nLangChain acts as the **"glue"** connecting the AI model to external tools, databases, APIs, and custom software.\n\n---\n\n### Core Concepts\n\n1. **Chains:** Link multiple steps or AI calls together. For instance, fetching information from a database $\\rightarrow$ summarizing it with an LLM $\\rightarrow$ emailing the summary.\n2. **Data Integration (RAG):** Allows LLMs to connect to external data sources (PDFs, websites, databases) so the model can answer questions using specific context.\n3. *

In [15]:

import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import Tool
from langchain.agents import create_agent


# 1. Get Gemini API key

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


# 2. Initialize Gemini LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)


# 3. Initialize Gemini Embeddings

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)


# 4. Create FAISS Vector Store

vector_store = FAISS.from_texts(
    ["initial memory"],
    embedding
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# 5. Create memory retrieval and storage functions

def retrieve_memory(query):
    documents = retriever.invoke(query)

    if not documents:
        return "No relevant previous conversation found."

    return "\n".join(doc.page_content for doc in documents)


def save_memory(query, answer):
    memory_text = f"User: {query}\nAssistant: {answer}"

    vector_store.add_texts([memory_text])


# 6. Create memory search tool

memory_tool = Tool(
    name="Search_Past_Memory",
    func=retrieve_memory,
    description=(
        "Search previously stored conversation memories. "
        "Use this tool when the user refers to earlier topics "
        "or asks about previous conversations."
    )
)


# 7. Create QA tool

def run_qa(question):
    response = llm.invoke(
        f"Answer clearly and accurately: {question}"
    )
    return response.content


qa_tool = Tool(
    name="Simple_QA",
    func=run_qa,
    description="Answer general factual and conceptual questions."
)


# 8. Create agent with tools

agent = create_agent(
    model=llm,
    tools=[qa_tool, memory_tool],
    system_prompt="""
    You are a helpful conversational AI assistant.

    Instructions:
    - Answer questions clearly and accurately.
    - Use Simple_QA for general factual questions.
    - Use Search_Past_Memory when previous conversation context is relevant.
    - Use retrieved memories to understand follow-up questions.
    - Do not invent information about previous conversations.
    """
)


# 9. Run multiple interactions

print("1️ First Question")

query1 = "What is LangChain?"

res1 = agent.invoke({
    "messages": [{"role": "user", "content": query1}]
})

answer1 = res1["messages"][-1].content
print("\nAnswer:", answer1)

save_memory(query1, answer1)


print("\n2️ Follow-up Question")

query2 = "Who created it?"

res2 = agent.invoke({
    "messages": [{"role": "user", "content": query2}]
})

answer2 = res2["messages"][-1].content
print("\nAnswer:", answer2)

save_memory(query2, answer2)


print("\n3️ Ask again about previous topic")

query3 = "Explain LangChain simply again."

res3 = agent.invoke({
    "messages": [{"role": "user", "content": query3}]
})

answer3 = res3["messages"][-1].content
print("\nAnswer:", answer3)

save_memory(query3, answer3)


1️ First Question

Answer: [{'type': 'text', 'text': "**LangChain** is an open-source development framework designed to simplify building applications powered by large language models (LLMs), such as OpenAI's GPT, Anthropic's Claude, or open-source models like Llama.\n\nInstead of only sending simple prompts and receiving text responses, LangChain provides abstractions and tooling to make LLM-powered applications **context-aware** and capable of **multi-step reasoning**.\n\n---\n\n### Core Concepts and Components\n\n1. **Model Abstractions & Prompt Templates:**\n   * Offers standardized interfaces to interact with various LLM providers, embedding models, and chat models.\n   * Manages dynamic, reusable prompt templates to ensure consistent input formats.\n\n2. **Chains:**\n   * Enables chaining together sequences of operations—for example, taking user input, fetching relevant documents, formatting a prompt, calling a model, and parsing the output into structured data (such as JSON).\n\

In [12]:
%pip install -q langchain-community